# Creating the matrix

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [ ]:
!pip install sentence-transformers

In [ ]:
data = pd.read_csv(r'movies.csv', delimiter=';')
data = data[data['country'] == 'USA']
data.dropna(subset=['original_title','description','genre'],inplace=True,axis=0)

data = data.reset_index(drop=True)

In [ ]:
data["combined"] = data['genre'] + ' ' + data['original_title'] + ' ' + data['description']
data.drop(['description','genre'],axis=1,inplace=True)

In [ ]:
model = SentenceTransformer('all-mpnet-base-v2')
movie_descriptions = data['combined'].tolist()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
%%time
movie_descriptions = data['combined'].tolist()
embeddings = model.encode(movie_descriptions)

CPU times: user 2h 35min 21s, sys: 29.2 s, total: 2h 35min 50s
Wall time: 3min 13s


In [ ]:
cosine_similarities = cosine_similarity(embeddings)

# Flask test

In [ ]:
!pip install flask pyngrok --quiet

In [ ]:
from flask import Flask, request, render_template
import requests
app = Flask(__name__, template_folder='/content/templates')

TMDB_API_KEY = '26cb56b9994cfcd57b70be461d5899f0'

def get_movie_poster(movie_title):
    search_url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query={movie_title}"
    response = requests.get(search_url)
    if response.status_code == 200:
        data = response.json()
        results = data.get('results')
        if results:
            poster_path = results[0].get('poster_path')
            if poster_path:
                return f"https://image.tmdb.org/t/p/w200{poster_path}"

    return None

def content_recommender_with_posters(title):
    movie_title = data['original_title']
    indices = pd.Series(data.index, index=data['original_title'])

    if title not in indices:
        raise ValueError("Movie not found.")

    idx = indices[title]
    sim_scores = list(enumerate(cosine_similarities[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    recommended_titles = movie_title.iloc[movie_indices].tolist()

    recommendations = []
    for t in recommended_titles:
        poster_url = get_movie_poster(t)
        recommendations.append({'title': t, 'poster_url': poster_url})

    return recommendations


@app.route('/', methods=['GET', 'POST'])
def index():
    recommendations = []
    movie_name = ''

    if request.method == 'POST':
        movie_name = request.form.get('movie')
        try:
            recommendations = content_recommender_with_posters(movie_name)
        except Exception as e:
            recommendations = [{'title': f"Error: {str(e)}", 'poster_url': None}]

    return render_template('index.html', recommendations=recommendations, movie_name=movie_name)

In [ ]:
from pyngrok import ngrok
import threading

ngrok.set_auth_token("2xB8rBCUFRz3bLHWyK4CSqmd3sV_2Urf5XxSwX49gWKSV2Rje")
public_url = ngrok.connect(5000)
print(f"Public URL: {public_url}")

def run_app():
    app.run()
thread = threading.Thread(target=run_app)
thread.start()

Public URL: NgrokTunnel: "https://0e8d-35-202-250-183.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use